# Learning curve: does the CNN branch pay off with more stations?

Step 1 of the transfer-learning plan. The CNN-LSTM and the LSTM-only model are trained on **growing numbers of training stations** and scored on the **same
test stations**. If the CNN-LSTM's advantage over the LSTM-only model grows with the number of stations (the gap rises toward and past 0), more stations, for example from
other Southeast Asian cities, can make the morphology branch pay off, and transfer learning on both the morphological and the temporal side is justified. If the gap stays flat
or negative at every size, the morphology adds little however many stations there are.

**How the experiment is run**

| | |
|---|---|
| training stations | `SIZES` (default 20, 40, 54 and all 67), picked by a seeded shuffle; the subsets of one draw are nested (20 inside 40 inside 54) |
| draws | `DRAWS` different random subsets per size below the full one (the full size runs once). Both models see the same subset of a draw, so the gap is paired |
| test stations | never change: the same ones for every point of the curve |
| GeoCP | off everywhere (no calibration stations) |
| epochs | fixed per model (`EPOCHS`: 7 for the CNN-LSTM, 15 for the LSTM-only, the epochs their cross-validation chose on the full data), cross-validation off, so every point costs one fit instead of four. `USE_CV = True` brings cross-validation back |
| what fewer stations also change | the spatial lag (the stand-in history of nearby stations) comes from the training stations only, so it gets sparser too. That is part of what a small set of stations means, and it affects both models |

**Resuming, and the 12-hour limit.** As in the main notebook: each step skips what is done, `TIME_BUDGET_H` stops the notebook cleanly before Kaggle ends the session, training
continues from its last finished fold when cross-validation is on. To continue: *Save Version*, then in a new session *Add Input* (this notebook's output and your data), run all.
The draws run in rounds (draw 0 of every size first), so a usable curve exists early and later rounds add the error bars.

| Cell | What it does |
|---|---|
| 1 | settings |
| 2 | helpers and checks |
| 3 | gets the repository, the data, the libraries (Kaggle only) |
| 4 | preprocess, train and report every point of the curve |
| 5 | the table and the two figures: test R² against stations, and the CNN-LSTM minus LSTM-only gap |

## 1. Settings (the only cell you edit)

In [ ]:
from pathlib import Path

ARM = "from_2023_with_preinstall"      # the dataset: data/Combined Dataset/<ARM> (this one keeps the Quezon City pre-installation readings)
RASTER = "master_raster_v3.tif"        # the morphology raster in data/Morphology (it must be in your data dataset)

SIZES = [20, 40, 54, None]              # training stations per point of the curve; None = all of them (67 here). Add more points, e.g. [10, 20, 30, 40, 54, None]
DRAWS = 3                               # random subsets per size below the full one; more draws = tighter error bars, more training time
MODELS = ["cnn", "nocnn"]               # cnn = CNN-LSTM, nocnn = LSTM only; both are needed for the gap
EPOCHS = {"cnn": 7, "nocnn": 15}        # fixed final-fit epochs (what cross-validation chose on the full data: 7 and 15). Used when USE_CV is False
USE_CV = False                          # False: no cross-validation, the fixed EPOCHS (one fit per point). True: spatial cross-validation picks the epochs, as in the main notebook (about 4x the time)
TIME_BUDGET_H = 11.0                    # stop by itself after this many hours (Kaggle ends a session at 12). Save Version, add it as an Input, run again to continue

STRIDE_NOTE = "no maps here: only the test accuracy is compared"
REDO = []                               # steps to run again although their output exists: any of "preprocess", "train", "report"
                                        # (redoing "preprocess" redoes the later steps too: a model only fits the data it was trained on)

DATA = Path("/kaggle/input/urban-air-data")                    # your data dataset: the exact path shown in the Input panel
GITHUB = "https://github.com/Patthethicc/urban-air.git"        # the code, used when no saved repository is attached
REFRESH_CODE = False                    # True: after restoring a saved repository, replace its code (not its runs) by the latest from GITHUB

## 2. Helpers and checks

In [ ]:
import json, os, shutil, subprocess, sys, time, traceback
import numpy as np, pandas as pd

T0 = time.time()                         # the clock of the time budget starts here (cell 2); a new session starts it again
ON_KAGGLE = Path("/kaggle/working").exists()
INPUT = Path("/kaggle/input")
if ON_KAGGLE:
    REPO = Path("/kaggle/working/urban-air")
else:                                    # a local checkout: open the notebook from the repository root
    REPO = Path.cwd() if (Path.cwd() / "model").is_dir() else Path.cwd().parent
MODEL = REPO / "model"
OUT = Path("/kaggle/working") if ON_KAGGLE else REPO       # where the table and the figure are written
STEPS = ("preprocess", "train", "report")
LABELS = {"cnn": "CNN-LSTM", "nocnn": "LSTM only (no CNN)"}
MIN_STEP_H = 0.1                         # a step is not started with less than this left of the time budget


class OutOfTime(Exception):
    """The time budget is used up: stop cleanly so that Save Version keeps everything."""


def need(*names):
    """A clear message instead of a NameError at the end of a long run: the names a cell relies on must exist."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"{', '.join(missing)} not defined: run the cells above in order (1 settings, 2 helpers, 3 setup) first")


def check_settings():
    need("ARM", "RASTER", "SIZES", "DRAWS", "MODELS", "EPOCHS", "USE_CV", "TIME_BUDGET_H", "REDO", "DATA", "GITHUB", "REFRESH_CODE")
    global REDO
    bad = [n for n in SIZES if n is not None and (not isinstance(n, int) or n < 6)]
    if bad or not SIZES:
        raise ValueError(f"SIZES must be whole numbers of at least 6 stations (or None for all of them), got {SIZES!r}")
    if len(set(SIZES)) != len(SIZES):
        raise ValueError(f"SIZES has a size twice: {SIZES!r}")
    if not isinstance(DRAWS, int) or DRAWS < 1:
        raise ValueError(f"DRAWS must be 1 or more, got {DRAWS!r}")
    if not MODELS or [m for m in MODELS if m not in LABELS]:
        raise ValueError(f"MODELS must be a list of {list(LABELS)}, got {MODELS!r}")
    if not USE_CV and [m for m in MODELS if m not in EPOCHS]:
        raise ValueError(f"EPOCHS needs an entry for each of {MODELS}, got {EPOCHS!r}")
    if not TIME_BUDGET_H > 0:
        raise ValueError(f"TIME_BUDGET_H must be above 0, got {TIME_BUDGET_H!r}")
    unknown = [r for r in REDO if r not in STEPS]
    if unknown:
        raise ValueError(f"REDO has unknown steps {unknown}; use any of {STEPS}")
    if "preprocess" in REDO:             # a model only fits the data it was trained on: redo everything after it
        REDO = sorted(set(REDO) | set(STEPS))


def left_h():
    """Hours left of the time budget."""
    return TIME_BUDGET_H - (time.time() - T0) / 3600


def points():
    """The (size, draw) points of the curve in the order they run: draw 0 of every size first, then draw 1, ... (the full size only once)."""
    return [(n, d) for d in range(DRAWS) for n in SIZES if not (n is None and d > 0)]


def pname(n, d):
    return f"n{'all' if n is None else n}_s{d}"


def overrides(n, d, model, extra=()):
    """Every stage needs the same overrides (the paths of this point and model, the station subset, the switches), so one function builds them."""
    base = f"runs/{ARM}/lc/{pname(n, d)}"
    kv = {"data.split.cv_assignment": "by_station", "geocp.enabled": "false",
          "model.spatial_cnn.enabled": "true" if model == "cnn" else "false",
          "data.split.train_subset.seed": d,
          "data.paths.master_raster": f"../data/Morphology/{RASTER}",
          "data.paths.sensor_locations_csv": f"../data/Combined Dataset/{ARM}/locations.csv",
          "data.paths.sensor_csv_dir": f"../data/Combined Dataset/{ARM}/sensors",
          "data.paths.processed_dir": f"{base}/processed", "data.paths.checkpoints_dir": f"{base}/checkpoints_{model}",
          "data.paths.snapshots_dir": f"{base}/snapshots_{model}", "data.paths.tuning_dir": f"{base}/tuning"}
    if n is not None:
        kv["data.split.train_subset.n_stations"] = n
    if not USE_CV:
        kv["train.cv.enabled"] = "false"
        kv["train.final_epochs"] = EPOCHS[model]
    return [x for k, v in kv.items() for x in ("--override", f"{k}={v}")] + list(extra)


def run(stage, n, d, model, *flags, extra=()):
    """One pipeline stage in its own process. The stage itself is stopped when the time budget is used up (exit code 124 -> OutOfTime)."""
    if left_h() < MIN_STEP_H:
        raise OutOfTime(f"before {stage}")
    budget = ["--override", f"train.runtime.max_hours={left_h():.3f}"]
    cmd = [sys.executable, "-u", "run_pipeline.py", "--stage", stage, *overrides(n, d, model, extra), *budget, *flags]
    print("$", " ".join(cmd), flush=True)
    t = time.time()
    code = subprocess.run(cmd, cwd=MODEL).returncode
    if code == 124:
        raise OutOfTime(f"during {stage}")
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)                           # stops the cell if a stage fails
    print(f"-- {stage}: {time.time() - t:.0f} s", flush=True)


def have(n, d, *parts):
    """Does runs/<ARM>/lc/<point>/<parts...> exist?"""
    return (MODEL / "runs" / ARM / "lc" / pname(n, d)).joinpath(*parts).exists()


def runs_of(n, d, model):
    """Finished training runs of one point and model (run_info.json says completed), oldest first."""
    out = []
    for info in sorted((MODEL / "runs" / ARM / "lc" / pname(n, d) / f"checkpoints_{model}").glob("*/run_info.json")):
        if json.loads(info.read_text(encoding="utf-8")).get("status") == "completed":
            out.append(info.parent.name)
    return out


def step(name, done, n, d, model, *flags, extra=()):
    """Run a pipeline stage unless its output already exists (and it is not in REDO)."""
    if done and name not in REDO:
        print(f"skip {name}: already done")
        return False
    run(name, n, d, model, *flags, extra=extra)
    return True


def status():
    """A line per point and model: what is finished."""
    rows = []
    for n, d in points():
        row = {"point": pname(n, d), "preprocess": "done" if have(n, d, "processed", "preprocess_info.json") else "-"}
        for m in MODELS:
            runs = runs_of(n, d, m)
            row[m] = "reported" if runs and have(n, d, f"snapshots_{m}", runs[-1], "metrics_summary.json") else ("trained" if runs else "-")
        rows.append(row)
    print(pd.DataFrame(rows).set_index("point").to_string())
    print(f"time used: {(time.time() - T0) / 3600:.1f} h of {TIME_BUDGET_H} h")


check_settings()
print("settings ok | dataset:", ARM, "| sizes:", SIZES, "| draws:", DRAWS, "| models:", MODELS, "| cross-validation:", USE_CV, "| repository:", REPO)

## 3. Get the repository, the data and the libraries
On Kaggle: if a saved repository is attached as an Input (a previous version of this notebook), it is restored, runs and all; otherwise the code is
cloned from GitHub. The data is copied from your data dataset (it is found even if the upload added a folder level). Locally this cell only checks that
everything is where it should be.

In [ ]:
check_settings()


def find_saved_repo():
    """A repository saved by an earlier session: any attached input that contains model/run_pipeline.py."""
    if INPUT.exists():
        for p in INPUT.glob("**/model/run_pipeline.py"):
            return p.parent.parent
    return None


def clone(into):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB, str(into)], check=True)


if ON_KAGGLE:
    if not (MODEL / "run_pipeline.py").is_file():
        saved = find_saved_repo()
        if saved is not None:
            shutil.copytree(saved, REPO, dirs_exist_ok=True)
            n = len(list((MODEL / "runs").glob("*/checkpoints*/*/run_info.json"))) if (MODEL / "runs").exists() else 0
            print(f"restored the repository from {saved} ({n} saved training run(s) found)")
        else:
            clone(REPO)
            print("cloned the repository from GitHub (no saved repository attached)")
    else:
        print("the repository is already in /kaggle/working")
    if REFRESH_CODE:                     # the latest code from GitHub; the saved runs and data are left as they are
        fresh = REPO.parent / "_fresh_code"
        shutil.rmtree(fresh, ignore_errors=True)
        clone(fresh)
        shutil.copytree(fresh / "model", MODEL, dirs_exist_ok=True, ignore=shutil.ignore_patterns("runs", "__pycache__", ".env"))
        shutil.rmtree(fresh, ignore_errors=True)
        print("code refreshed from GitHub")

    raster = REPO / "data" / "Morphology" / RASTER
    if not raster.is_file():             # the data is not in place yet: find it under DATA wherever the upload put it
        found = list(DATA.rglob(RASTER)) if DATA.exists() else []
        if found:
            src = found[0].parent.parent                      # the folder that holds Morphology/, Combined Dataset/, ...
            for part in (Path("Morphology"), Path("Boundaries"), Path("Combined Dataset") / ARM):
                if (src / part).is_dir():
                    shutil.copytree(src / part, REPO / "data" / part, dirs_exist_ok=True)
            print("data copied from", src)
        else:
            zips = list(DATA.rglob("*.zip")) if DATA.exists() else []
            if zips:                                          # Kaggle left an uploaded zip packed: its top-level `data/` folder lands in REPO
                import zipfile
                zipfile.ZipFile(zips[0]).extractall(REPO)
                print("data unzipped from", zips[0])

    need_pip = {"skopt": "scikit-optimize", "libpysal": "libpysal", "esda": "esda", "rasterio": "rasterio", "geopandas": "geopandas",
                "shapely": "shapely", "pyproj": "pyproj", "affine": "affine", "yaml": "pyyaml", "sklearn": "scikit-learn", "contextily": "contextily"}
    import importlib.util
    missing = [pip for mod, pip in need_pip.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("installing:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    old_esda = "import inspect; from esda.moran import Moran_Local as M; raise SystemExit(0 if 'alternative' in inspect.signature(M.__init__).parameters else 1)"
    if subprocess.run([sys.executable, "-c", old_esda]).returncode != 0:      # the spatial-bias check needs the two-sided LISA option
        print("esda is too old: upgrading")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "esda", "libpysal"], check=True)

# the checks that stop a run in its first seconds instead of after hours
problems = []
if not (MODEL / "run_pipeline.py").is_file():
    problems.append(f"{MODEL / 'run_pipeline.py'} is missing: no repository (attach a saved one, or check GITHUB and the internet switch)")
if not (REPO / "data" / "Morphology" / RASTER).is_file():
    problems.append(f"data/Morphology/{RASTER} is missing: put it in the Morphology folder of your data dataset (DATA = {DATA})")
if not (REPO / "data" / "Combined Dataset" / ARM / "locations.csv").is_file():
    problems.append(f"data/Combined Dataset/{ARM}/locations.csv is missing: check ARM and your data dataset")
if problems:
    raise SystemExit("cannot start:\n  - " + "\n  - ".join(problems))
print("ready | state of each point of the curve:")
status()

## 4. Train and report every point of the curve
For each point (a number of training stations and a draw), in rounds: preprocess once (the subset, the spatial lag and the scalers belong to the point), then train and report each model in `MODELS`.
Each step is skipped if its output is already there. When `TIME_BUDGET_H` is used up the notebook stops cleanly and prints what to do (Save Version, add the output as an Input, run again).

In [ ]:
check_settings()


def run_point(n, d):
    print("=" * 78 + f"\n{pname(n, d)}: {'all' if n is None else n} training stations, draw {d}  ({left_h():.1f} h of the time budget left)\n" + "=" * 78)
    first = MODELS[0]
    step("preprocess", have(n, d, "processed", "scalers.json") and have(n, d, "processed", "preprocess_info.json"), n, d, first)
    for model in MODELS:
        print(f"--- {LABELS[model]}")
        if "train" in REDO or not runs_of(n, d, model):
            run("train", n, d, model, extra=["--override", "train.resume=true"])         # continues a run that was stopped, or starts a new one
        run_id = runs_of(n, d, model)[-1]
        step("report", have(n, d, f"snapshots_{model}", run_id, "metrics_summary.json"), n, d, model, "--run-id", run_id)


try:
    for n, d in points():
        run_point(n, d)
    print("\nevery point of the curve is finished")
except OutOfTime as stop:
    print(f"\nTIME BUDGET REACHED ({stop}) after {(time.time() - T0) / 3600:.1f} h. Nothing is lost: finished steps are saved.")
    print("To continue: Save Version, then in a new session Add Input (this notebook's output + your data), and run all again.")
except Exception:
    traceback.print_exc()
    print("A step failed (everything finished so far is saved). Fix the cause above and run this cell again: finished steps are skipped.")
finally:
    status()

## 5. The learning curve
Test R² of both models against the number of training stations (a dot per draw, a line through the means), and the gap CNN-LSTM minus LSTM-only. The gap is paired: both models of a draw
saw the same stations. **Reading it:** if the gap rises with the number of stations and its mean gets above 0, more stations can make the morphology branch pay off; if it stays flat around or below 0, it cannot.

In [ ]:
try:
    need("MODELS", "SIZES")
    import matplotlib
    import matplotlib.pyplot as plt

    rows = []
    for n, d in points():
        for m in MODELS:
            runs = runs_of(n, d, m)
            metrics = MODEL / "runs" / ARM / "lc" / pname(n, d) / f"snapshots_{m}" / (runs[-1] if runs else "-") / "metrics_summary.json"
            info = MODEL / "runs" / ARM / "lc" / pname(n, d) / "processed" / "preprocess_info.json"
            if not (runs and metrics.is_file() and info.is_file()):
                continue
            t = json.loads(metrics.read_text(encoding="utf-8"))["accuracy"]["measures"]["test"]
            stations = json.loads(info.read_text(encoding="utf-8"))["stations_per_partition"]["train"]
            rows.append({"point": pname(n, d), "stations": stations, "draw": d, "model": m, "test_R2": t["r2"], "test_RMSE": t["rmse"],
                         "test_MAE": t["mae"], "test_rows": t["rows"], "test_stations": t["stations"], "run": runs[-1]})
    table = pd.DataFrame(rows)
    if table.empty:
        print("no finished point yet: run cell 4 first")
    else:
        table.round(4).to_csv(OUT / "learning_curve.csv", index=False)
        wide = table.pivot_table(index=["stations", "draw"], columns="model", values="test_R2")
        summary = table.groupby(["stations", "model"])["test_R2"].agg(["mean", "std", "min", "max", "count"]).round(4)
        print(summary.to_string())
        gap = None
        if {"cnn", "nocnn"} <= set(wide.columns):
            gap = (wide["cnn"] - wide["nocnn"]).dropna().rename("gap").reset_index()
            by = gap.groupby("stations")["gap"].agg(["mean", "std", "min", "max", "count"]).round(4)
            print("\nCNN-LSTM minus LSTM-only test R2, paired by draw:")
            print(by.to_string())
            by.to_csv(OUT / "learning_curve_gap.csv")
            if gap["stations"].nunique() >= 2:
                slope = np.polyfit(gap["stations"], gap["gap"], 1)[0] * 10
                print(f"\ntrend of the gap: {slope:+.4f} R2 per +10 stations (a line through all draws)")
        fig, axes = plt.subplots(1, 2 if gap is not None else 1, figsize=(13 if gap is not None else 7, 4.8), squeeze=False)
        ax = axes[0, 0]
        colours = {"cnn": "#2B66B1", "nocnn": "#D2691E"}
        for m in MODELS:
            sub = table[table["model"] == m]
            ax.scatter(sub["stations"], sub["test_R2"], s=28, color=colours[m], alpha=0.45, linewidths=0)
            mean = sub.groupby("stations")["test_R2"].mean()
            ax.plot(mean.index, mean.values, "-o", color=colours[m], label=LABELS[m])
        ax.set_xlabel("training stations")
        ax.set_ylabel("test R2 (same test stations)")
        ax.set_title("Test accuracy against the number of training stations", fontsize=10, loc="left")
        ax.legend(frameon=False)
        ax.grid(alpha=0.25)
        if gap is not None:
            ax = axes[0, 1]
            ax.axhline(0, color="#17212B", linewidth=1)
            ax.scatter(gap["stations"], gap["gap"], s=28, color="#2B66B1", alpha=0.45, linewidths=0)
            mean = gap.groupby("stations")["gap"].mean()
            ax.plot(mean.index, mean.values, "-o", color="#2B66B1")
            ax.set_xlabel("training stations")
            ax.set_ylabel("CNN-LSTM minus LSTM-only, test R2")
            ax.set_title("Above 0: the CNN branch helps", fontsize=10, loc="left")
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(OUT / "learning_curve.png", dpi=150)
        plt.show()
        print("written:", OUT / "learning_curve.csv", "|", OUT / "learning_curve.png")
except Exception:
    traceback.print_exc()
    print("The curve failed; each point's report is runs/<dataset>/lc/<point>/snapshots_<model>/<run>/SUMMARY.md")